# Access SOFAR data from HPSS archive, convert BUFR files to netCDF, and compute matchups between GFS and SOFAR data

In [1]:
import subprocess
import os
import glob
import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import netCDF4 as nc
import xarray as xr
import ncepbufr

In [2]:
%load_ext autoreload
%autoreload 2

import ww3tools.sofar as sf

## Access the SOFAR data from the NOAA HPSS system for given dates


In [ ]:
# Define start and end dates 
start_date = "2026-01-01"
end_date = "2026-01-31"

# Define the working data directory to save the data
working_dir = "/scratch3/NCEPDEV/marine/Edwin.Rainville/ww3-tools-dev/data/bufr_data/"

# Define slurm parameters
slurm_params = {
            "ntasks": 1,
            "queue": "windfall",
            "partition": "u1-service",
            "walltime": "08:00:00",
            "account": "marine-cpu",
            "job_name": "hpss_sofar"
            }

# Access the data
sf.get_sofar_hpss(start_date, end_date, slurm_params, working_dir)

## Convert the BUFR Format files to netCDF4 file formats

In [15]:
# Define paths on Ursa
bufr_dir = '/scratch3/NCEPDEV/marine/Edwin.Rainville/ww3-tools-dev/data/bufr_data'
output_dir = '/scratch3/NCEPDEV/marine/Edwin.Rainville/ww3-tools-dev/data/nc_data'

start_date = "2026-01-01"
end_date = "2026-01-31"

# Run batch conversion
nc_files = sf.convert_sofar_bufr_to_nc(
    bufr_dir=bufr_dir,
    output_dir=output_dir,
    start_date=start_date,
    end_date=end_date,
    pattern="sofar_b001_xx121_*"
)

print(f"Generated {len(nc_files)} NetCDF files.")

Found 70 BUFR files to process.
  [sofar_b001_xx121_20251224] No records fell within date range [2026-01-01 to 2026-01-31]. Skipping.
  [sofar_b001_xx121_20260101] Saved CF-1.9 compliant NetCDF (5730 records) -> sofar_b001_xx121_20260101.nc
  [sofar_b001_xx121_20260102] Saved CF-1.9 compliant NetCDF (5738 records) -> sofar_b001_xx121_20260102.nc
  [sofar_b001_xx121_20260103] Saved CF-1.9 compliant NetCDF (5734 records) -> sofar_b001_xx121_20260103.nc
  [sofar_b001_xx121_20260104] Saved CF-1.9 compliant NetCDF (5781 records) -> sofar_b001_xx121_20260104.nc
  [sofar_b001_xx121_20260105] Saved CF-1.9 compliant NetCDF (5796 records) -> sofar_b001_xx121_20260105.nc
  [sofar_b001_xx121_20260106] Saved CF-1.9 compliant NetCDF (5810 records) -> sofar_b001_xx121_20260106.nc
  [sofar_b001_xx121_20260107] Saved CF-1.9 compliant NetCDF (5815 records) -> sofar_b001_xx121_20260107.nc
  [sofar_b001_xx121_20260108] Saved CF-1.9 compliant NetCDF (5859 records) -> sofar_b001_xx121_20260108.nc
  [sofar_b

In [22]:
ds = xr.open_dataset('/scratch3/NCEPDEV/marine/Edwin.Rainville/ww3-tools-dev/data/nc_data/sofar_b001_xx121_20260101.nc')
ds

<xarray.Dataset> Size: 688kB
Dimensions:                                                    (obs: 5730)
Coordinates:
  * obs                                                        (obs) int64 46kB ...
    time                                                       (obs) datetime64[ns] 46kB ...
Data variables:
    platform_id                                                (obs) <U8 183kB ...
    latitude                                                   (obs) float64 46kB ...
    longitude                                                  (obs) float64 46kB ...
    sea_surface_wave_significant_height                        (obs) float64 46kB ...
    wind_from_direction                                        (obs) float64 46kB ...
    wind_speed                                                 (obs) float64 46kB ...
    sea_surface_wave_directional_spread                        (obs) float64 46kB ...
    sea_surface_wave_from_direction_at_variance_spectral_peak  (obs) float64 46kB ...
    sea_surface_wave_mean_period                               (obs) float64 46kB ...
    sea_surface_wave_period_at_variance_spectral_peak          (obs) float64 46kB ...
Attributes:
    Conventions:  CF-1.9
    featureType:  trajectory
    title:        Surface Marine Observation Data Converted from NCEP BUFR
    source:       NCEP BUFR observation file: sofar_b001_xx121_20260101
    history:      2026-10-06T15:29:44Z: Converted to CF-1.9 NetCDF via ww3-to...

In [23]:
ds.close()

## Combine Sofar data and set up for Interpolations

In [7]:
# Get SOFAR netCDF file names
sofar_files = sorted(glob.glob("/scratch3/NCEPDEV/marine/Edwin.Rainville/ww3-tools-dev/data/nc_data/sofar_*.nc"))
chunk_size = {"time": 500}

ds_sofar = xr.open_mfdataset(
   sofar_files,
   combine="nested",
   concat_dim= "obs",
   parallel=True,
   chunks={"obs": 500},
   engine="netcdf4",
)

# Ensure coordinates are properly sorted by time
ds_sofar = ds_sofar.sortby("time")

ds_sofar


<xarray.Dataset> Size: 22MB
Dimensions:                                                    (obs: 180369)
Coordinates:
  * obs                                                        (obs) int64 1MB ...
    time                                                       (obs) datetime64[ns] 1MB dask.array<chunksize=(477,), meta=np.ndarray>
Data variables:
    platform_id                                                (obs) <U8 6MB dask.array<chunksize=(477,), meta=np.ndarray>
    latitude                                                   (obs) float64 1MB dask.array<chunksize=(477,), meta=np.ndarray>
    longitude                                                  (obs) float64 1MB dask.array<chunksize=(477,), meta=np.ndarray>
    sea_surface_wave_significant_height                        (obs) float64 1MB dask.array<chunksize=(477,), meta=np.ndarray>
    wind_from_direction                                        (obs) float64 1MB dask.array<chunksize=(477,), meta=np.ndarray>
    wind_speed                                                 (obs) float64 1MB dask.array<chunksize=(477,), meta=np.ndarray>
    sea_surface_wave_directional_spread                        (obs) float64 1MB dask.array<chunksize=(477,), meta=np.ndarray>
    sea_surface_wave_from_direction_at_variance_spectral_peak  (obs) float64 1MB dask.array<chunksize=(477,), meta=np.ndarray>
    sea_surface_wave_mean_period                               (obs) float64 1MB dask.array<chunksize=(477,), meta=np.ndarray>
    sea_surface_wave_period_at_variance_spectral_peak          (obs) float64 1MB dask.array<chunksize=(477,), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.9
    featureType:  trajectory
    title:        Surface Marine Observation Data Converted from NCEP BUFR
    source:       NCEP BUFR observation file: sofar_b001_xx121_20260101
    history:      2026-10-06T15:29:44Z: Converted to CF-1.9 NetCDF via ww3-to...

In [10]:
# 4. Check original longitude range
lon_min = ds_sofar.longitude.min().compute().item()
lon_max = ds_sofar.longitude.max().compute().item()
print(f"Original longitude range: {lon_min:.2f} to {lon_max:.2f}")

# 5. Convert longitude from [-180, 180] to [0, 360] format
ds_sofar["longitude"] = (ds_sofar["longitude"] + 360) % 360

# 6. Verify the shift worked
new_lon_min = ds_sofar.longitude.min().compute().item()
new_lon_max = ds_sofar.longitude.max().compute().item()
print(f"New longitude range: {new_lon_min:.2f} to {new_lon_max:.2f}")

Original longitude range: 0.01 to 360.00
New longitude range: 0.01 to 360.00


In [ ]:
gfs_path = '/scratch3/NCEPDEV/climate/Jessica.Meixner/Data/retrov17_01'